In [0]:
%sql
SELECT DISTINCT
  table_catalog,
  table_schema,
  table_name
FROM 
  system.information_schema.columns
WHERE  
  table_schema NOT IN ('access', 'default', 'information_schema')
ORDER BY 2, 3, 1

In [0]:
%sql
SELECT 
  table_catalog AS `catalog`,
  table_schema AS `schema`,
  table_name,
  column_name,
  ordinal_position AS `column_ordinal_position`,
  full_data_type AS `column_data_type`
FROM 
  system.information_schema.columns
WHERE
  table_schema NOT IN ('access', 'default', 'information_schema')
  AND table_catalog = 'silver_dev'
ORDER BY
  table_catalog,
  table_schema,
  table_name,
  ordinal_position

In [0]:
import pandas as pd

df = _sqldf.toPandas()
df

In [0]:
pk_map = {
    ("geo", "counties"): {"county_fips"},
    ("geo", "states"): {"state_fips"},
    ("fact", "population"): {"county_fips", "year"},  # composite PK
    ("dim", "providers"): {"npi"}
}


fk_map = [
    # (from_catalog, from_schema, from_table, from_column,
    #  to_catalog,   to_schema,   to_table,   to_column)

    ("silver_dev", "census_bureau", "county_pop", "county_fips",
     "silver_dev", "geo", "counties", "county_fips"),

    ("silver_dev", "geo", "counties", "state_fips",
     "silver_dev", "geo", "states", "state_fips"),
    
    ("silver_dev", "cms", "hospital_general_information", "county_fips",
     "silver_dev", "geo", "counties", "county_fips"),
    
    ("silver_dev", "economic_research_service", "poverty", "county_fips",
     "silver_dev", "geo", "counties", "county_fips"),
        
    ("silver_dev", "ruca", "ruca_census_tract", "county_fips_23",
     "silver_dev", "geo", "counties", "county_fips"),

    # Dim: Poverty attributes
    ("silver_dev", "economic_research_service", "poverty", "attribute_name",
     "silver_dev", "economic_research_service", "poverty_attributes", "attribute_name"),
    
    # Dim: RUCA Codes
    ("silver_dev", "ruca", "ruca_census_tract", "primary_ruca",
     "silver_dev", "ruca", "ruca_codes", "ruca_code"),
    
    ("silver_dev", "ruca", "ruca_census_tract", "secondary_ruca",
     "silver_dev", "ruca", "ruca_codes", "ruca_code"),
]

In [0]:


# Optional: sort to ensure correct column order
df = df.sort_values(
    by=["catalog", "schema", "table_name", "column_ordinal_position"]
)

dbml_lines = []

for (schema, table), group in df.groupby(["schema", "table_name"]):
    table_name = f"{schema}.{table}"
    dbml_lines.append(f"Table {table_name} {{")
    
    pk_columns = pk_map.get((schema, table), set())
    
    for _, row in group.iterrows():

        # For now, force lower column name because it's not updating NPI column names
        col_name = row["column_name"].lower()
        col_type = row["column_data_type"]
        
        if col_name in pk_columns:
            dbml_lines.append(f"  {col_name} {col_type} [primary key]")
        else:
            dbml_lines.append(f"  {col_name} {col_type}")
    
    dbml_lines.append("}\n")


# Add foreign key references
for (
    from_catalog, from_schema, from_table, from_col,
    to_catalog, to_schema, to_table, to_col
) in fk_map:
    
    #from_ref = f"{from_catalog}.{from_schema}.{from_table}.{from_col}"
    #to_ref   = f"{to_catalog}.{to_schema}.{to_table}.{to_col}"
    from_ref = f"{from_schema}.{from_table}.{from_col}"
    to_ref   = f"{to_schema}.{to_table}.{to_col}"
    
    dbml_lines.append(f"Ref: {to_ref} < {from_ref}")

dbml_output = "\n".join(dbml_lines)

print(dbml_output)

# Future Addition: Pull primary keys from the tables

In [0]:
%skip
%sql
SELECT
  kcu.table_catalog,
  kcu.table_schema,
  kcu.table_name,
  kcu.column_name
FROM
  system.information_schema.table_constraints tc
JOIN
  system.information_schema.key_column_usage kcu
ON
  tc.constraint_name = kcu.constraint_name
  AND tc.table_schema = kcu.table_schema
  AND tc.table_name = kcu.table_name
WHERE
  tc.constraint_type = 'PRIMARY KEY'
  AND kcu.table_catalog = 'silver'